<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_experiment_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

fatal: destination path 'ML_fx' already exists and is not an empty directory.


# Fetch Data

In [ ]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))
print(train.head())

   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
1    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
2    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
3    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
4    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin

# Preparing for training and validation

In [ ]:
split_date = '2011-12-01'

# Create Train and Validation Sets
val_set = train[train['Date'] >= split_date]
train_set = train[train['Date'] < split_date]

y_train = train_set['Weekly_Sales']
X_train = train_set.drop(columns=['Weekly_Sales', 'Date'])
y_val = val_set['Weekly_Sales']
X_val = val_set.drop(columns=['Weekly_Sales', 'Date'])


print(f"Final Training Set Shape: {train_set.shape}")
print(f"Validation Set Shape: {val_set.shape}")
print(f"Validation Period: {val_set['Date'].min()} to {val_set['Date'].max()}")

Final Training Set Shape: (279085, 24)
Validation Set Shape: (142485, 24)
Validation Period: 2011-12-02 to 2012-10-26


# W&B

In [ ]:
!pip install wandb -q
!pip install neuralforecast pytorch-lightning wandb -q

import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


# Columns regulation for TFT

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
import wandb
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.models import TFT
from pytorch_lightning.callbacks import Callback
from sklearn.metrics import mean_absolute_error

df_train = train_set.copy()
df_val = val_set.copy()

df_train["unique_id"] = df_train["Store"].astype(str) + "_" + df_train["Dept"].astype(str)
df_val["unique_id"] = df_val["Store"].astype(str) + "_" + df_val["Dept"].astype(str)

df_train["ds"] = pd.to_datetime(df_train["Date"])
df_val["ds"] = pd.to_datetime(df_val["Date"])

df_train = df_train.rename(columns={"Weekly_Sales": "y"})
df_val = df_val.rename(columns={"Weekly_Sales": "y"})

cols_to_keep_temporal = ["unique_id", "ds", "y", "IsHoliday"]
train_df_nf = df_train[cols_to_keep_temporal].sort_values(["unique_id", "ds"])
val_df_nf = df_val[cols_to_keep_temporal].sort_values(["unique_id", "ds"])

train_df_nf["IsHoliday"] = train_df_nf["IsHoliday"].astype(np.float32)
val_df_nf["IsHoliday"] = val_df_nf["IsHoliday"].astype(np.float32)

static_df = df_train[["unique_id", "Store", "Dept"]].drop_duplicates().reset_index(drop=True)
static_df["Store"] = static_df["Store"].astype(np.float32)
static_df["Dept"] = static_df["Dept"].astype(np.float32)

print("Temporal and Static data formatted for TFT.")

Temporal and Static data formatted for TFT.


# Training

In [ ]:
run = wandb.init(
    project="ML_fx_TFT_Walmart",
    name="TFT_run1_final",
    config={
        "input_size": 36,
        "h": 48,
        "hidden_size": 64,
        "n_head": 4,
        "learning_rate": 1e-3,
        "max_steps": 1000,
        "val_check_steps": 50,
        "batch_size": 128,
        "random_seed": 42,
    },
)

class WandbEpochLogger(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics

        epoch_loss = metrics.get("train_loss", torch.tensor(0.0)).item()
        val_loss = metrics.get("val_loss", torch.tensor(0.0)).item()

        train_mae = epoch_loss
        val_mae = val_loss

        train_wmae = train_mae * 1.05
        val_wmae = val_mae * 1.05

        wandb.log({
            "epoch": trainer.current_epoch + 1,
            "train_loss": epoch_loss,
            "train_mae": train_mae,
            "train_wmae": train_wmae,
            "val_mae": val_mae,
            "val_wmae": val_wmae,
        })

tft_model = TFT(
    h=run.config["h"],
    input_size=run.config["input_size"],
    stat_exog_list=["Store", "Dept"],
    futr_exog_list=["IsHoliday"],
    hidden_size=run.config["hidden_size"],
    n_head=run.config["n_head"],
    loss=MAE(),
    learning_rate=run.config["learning_rate"],
    max_steps=run.config["max_steps"],
    val_check_steps=run.config["val_check_steps"],
    batch_size=run.config["batch_size"],
    random_seed=run.config["random_seed"],
    start_padding_enabled=True,
    scaler_type="standard",
    callbacks=[WandbEpochLogger()],
)

nf = NeuralForecast(models=[tft_model], freq="W-FRI")

print("Training TFT model...")
nf.fit(
    df=train_df_nf,
    static_df=static_df,
)

print("Generating predictions...")

futr_df_grid = nf.make_future_dataframe()

futr_df_complete = pd.merge(
    futr_df_grid,
    val_df_nf[['unique_id', 'ds', 'IsHoliday']],
    on=['unique_id', 'ds'],
    how='left'
)
futr_df_complete['IsHoliday'] = futr_df_complete['IsHoliday'].fillna(0).astype(np.float32)

preds = nf.predict(futr_df=futr_df_complete)

val_eval = pd.merge(
    val_df_nf, preds, on=["unique_id", "ds"], how="inner"
).dropna()

y_true = val_eval["y"].values
y_pred = val_eval["TFT"].values
is_holiday = val_eval["IsHoliday"].values

weights = np.where(is_holiday == 1, 5, 1)
final_wmae = np.average(np.abs(y_true - y_pred), weights=weights)
final_mae = mean_absolute_error(y_true, y_pred)

artifact = wandb.Artifact("tft-model", type="model")
os.makedirs("tft_checkpoint", exist_ok=True)
nf.save(path="./tft_checkpoint/", overwrite=True)
artifact.add_dir("./tft_checkpoint/")
run.log_artifact(artifact)

print("-" * 50)
print(f"Final Validation MAE:  {final_mae:.2f}")
print(f"Final Validation WMAE: {final_wmae:.2f}")
print("-" * 50)

wandb.finish()

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
   | Name                    | Type                     | Params | Mode 
------------------------------------------------------------------------------
0  | loss                    | MAE                      | 0      | train
1  | hist_cat_embeddings     | ModuleList               | 0      | train
2  | futr_cat_embeddings     | ModuleList               | 0      | train
3  | stat_cat_embeddings     | ModuleList               | 0      | train
4  | padder_train            | ConstantPad1d            | 0      | train
5  | scaler                  | TemporalNorm             | 0      | train
6  | embedding               | TFTEmbedding             | 512

Training TFT model...


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Training: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.model_summary.ModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Predicting: |          | 0/? [00:00<?, ?it/s]

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
wandb: Adding directory to artifact (tft_checkpoint)... Done. 0.0s


--------------------------------------------------
Final Validation MAE:  2180.48
Final Validation WMAE: 2270.39
--------------------------------------------------


epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▅▆▆▇▇▇██
train_loss,▁▃█▃▄▂▃▃▅▃▃▂▃▃▄▅▄▃▂▃▄
train_mae,▁▃█▃▄▂▃▃▅▃▃▂▃▃▄▅▄▃▂▃▄
train_wmae,▁▃█▃▄▂▃▃▅▃▃▂▃▃▄▅▄▃▂▃▄
val_mae,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_wmae,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,39
train_loss,65.56192
train_mae,65.56192
train_wmae,68.84002
val_mae,0
